In [1]:
import pandas as pd

In [2]:
wldf = pd.read_excel(r"c:\Users\glj523\Downloads\Wetlab Summary_Export to Magnus.xlsx", dtype=str)
asdf = pd.read_excel(r"c:\Users\glj523\Downloads\SedimentArchive&SubSamplingALL_231214 (3).xlsx", dtype=str)
kurt = pd.read_excel(r"c:\Users\glj523\Downloads\ISL_Robot_27.11.2023.xlsx", sheet_name="ALL", dtype=str)
cgg = pd.read_excel(r"c:\Users\glj523\Downloads\CGG_Database_Sediment_Water_231115_PSO (1).xlsx", sheet_name="Sediment_Water", dtype=str)
iceland_master_cores = pd.read_excel(r"c:\Users\glj523\Downloads\ROCS_lake_network.xlsx", dtype=str)

In [3]:
pd.set_option('display.max_rows', 1000)  # Set maximum number of rows to display
pd.set_option('display.max_columns', 1000)

In [7]:
asdf = asdf.drop(columns="No")
cgg = cgg.dropna(subset="Museum ID/sample ID")
asdf = asdf.dropna(how="all")
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.upper()

In [6]:
sorted(list(cgg[cgg["Museum ID/sample ID"].str.lower().str.startswith("gel")]["Museum ID/sample ID"].unique()))

['Gel lab 23.1.14']

Filtering out all rows that doesnt have DepthSampledCalTape because those cannot be cores

In [8]:
cores = asdf[~asdf["DepthSampledCalTape"].isna()]

Cannot find cores T1A, T1B,T3B. Only found these that start with T

In [18]:
cores[cores["BulkSampleID"].str.lower().str.startswith("t")]["BulkSampleID"].unique()

array(['TDV1.1', 'TÅRUPLUND_1', 'TÅRUPLUND_2'], dtype=object)

In [19]:
cores[cores["BulkSampleID"].str.lower().str.startswith("g")]["BulkSampleID"].unique()

array(['GELD_3.1', 'GELD_4.2', 'GOR22SS', 'GOR22A_1', 'GOR22A_2',
       'GOR22A_3', 'GOR22A_4', 'GOR22A_5', 'GOR22A_6', 'GOR22A_7',
       'GOR22A_8', 'GOR22A_9', 'GOR22A_10', 'GOR22A_11', 'GOR22A_12',
       'GOR22A_13', 'GOR22A_14', 'GOR22A_15'], dtype=object)

In [20]:
cores[cores["BulkSampleID"].str.lower().str.startswith("h")]["BulkSampleID"].unique()

array(['HFK8', 'HOLLERUP12 HOL23012', 'HOLLERUP11 HOL23011',
       'HOLLERUP10 HOL23010', 'HOLLERUP9 HOL23009', 'HOLLERUP8 HOL23008',
       'HOLLERUP7 HOL23007', 'HOLLERUP6 HOL23006', 'HOLLERUP5 HOL23005',
       'HOLLERUP 4 HOL23004', 'HOLLERUP 3 HOL23003',
       'HOLLERUP 2 HOL23002', 'HOLLERUP 1 HOL23001'], dtype=object)

However i found many that starts with "ISL":

In [10]:
iceland_cores = cores[cores["BulkSampleID"].str.startswith("ISL")]

In [11]:
iceland_cores["BulkSampleID"].unique()

array(['ISL22-20', 'ISL22-18', 'ISL21_12_1', 'ISL21_12_2', 'ISL21_11',
       'ISL21_14B', 'ISL22-20.7', 'ISL22-20.2A', 'ISL22-20.2B',
       'ISL22-20.5', 'ISL22-20.3A', 'ISL22-20.3B', 'ISL22-20.6',
       'ISL21-15A', 'ISL21-10A', 'ISL22-24', 'ISL22-27', 'ISL22-48-1',
       'ISL22-48-5', 'ISL21-13', 'ISL22-48.2', 'ISL22-48.6', 'ISL2020-02',
       'ISL22-43', 'ISL22-47', 'ISL22-46', 'ISL22-36', 'ISL22-52',
       'ISL22-21', 'ISL22-53', 'ISL22-17', 'ISL22-40.6', 'ISL22-40.2',
       'ISL22-40.3', 'ISL22-40.4', 'ISL22-41', 'ISL22-42', 'ISL22-55',
       'ISL22-35.9', 'ISL22-35.8', 'ISL22-35.3', 'ISL22-35.4',
       'ISL22-35.2', 'ISL22-35.7', 'ISL22-35.6', 'ISL22-49.1',
       'ISL22-49.2', 'ISL22-49.3', 'ISL22-49.4', 'ISL22-49.6',
       'ISL22-49.5', 'ISL22-49.7', 'ISL22-49.8', 'ISL22-49.9',
       'ISL22-49.10', 'ISL22-37.3', 'ISL22-37.1', 'ISL22-37.4',
       'ISL22-37.2', 'ISL22-33.10', 'ISL22-33.4', 'ISL22-33.9',
       'ISL22-33.3', 'ISL22-33.2', 'ISL22-33.7', 'ISL22-33.1',
  

Count

In [12]:
len(iceland_cores["BulkSampleID"].unique())

115

Merging on Archive Sample ID. Caution: Xihan says this ID is not robust in her data, so maybe this is error prone? Think it should be fine as long as its LV codes however. Best would be to merge with Robot Sample ID, will do so when I get the data from Jesper.

In [13]:
merged_df = pd.merge(iceland_cores, wldf, left_on='ArchiveSampleID', right_on='Archive Sample ID', how='inner')

In [14]:
merged_df_essentials = merged_df[['BulkSampleID', "ArchiveSampleID", "Robot Sample ID", "Library ID", 'FastQ File ID', "DepthSampledCalTape"]]

In [15]:
merged_df_essentials.to_csv("iceland_cores_essential_metadata.csv")

In [16]:
kurt_data = merged_df_essentials[merged_df_essentials["BulkSampleID"].isin(kurt["Sample Type"].dropna().unique())]

In [17]:
kurt_data.to_csv("kurt_iceland_essentials.csv")